# Coop Case — Q1: Channel Behavior

**Question:** How do offline-, online- and omni-channel customers shop, and what sets them apart?

Sub-questions we'll answer:
- Do the channels differ in basket value / basket size?
- Do they shop at different times of day?
- Do online customers buy less of "luxury" / premium items (can't see them physically)?
- Do omni-channel customers spend more overall than single-channel customers?
- How common is each channel / how many households are omni-channel?


## 1. Setup & load data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 50)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)


In [ ]:
DATA_PATH = "2months_v2/rl_2months.csv"

dtypes = {
    "receiptKey": "int64",
    "hourOfDay": "int8",
    "minuteOfHour": "int8",
    "quantity": "float32",
    "lineItemAmount": "float32",
    "lineItemAmountExclVat": "float32",
    "discountAmountExclVat": "float32",
    "lineItemCostExclVat": "float32",
    "CoopOnlineYN": "category",
    "store": "category",
    "customerId": "Int64",
    "householdId": "Int64",
    "MOSAICGroup": "category",
    "MOSAICGroupDescription": "category",
    "MOSAICType": "category",
    "MOSAICTypeDescription": "category",
    "DominantBuyingPowerClass": "category",
    "ItemID": "int64",
    "ItemSubSegmentName": "category",
    "ItemSubSegmentID": "Int64",
    "ItemSegmentName": "category",
    "ItemSegmentID": "Int64",
    "ItemSubCategoryName": "category",
    "ItemSubCategoryID": "Int64",
    "ItemCategoryName": "category",
    "ItemCategoryID": "Int64",
    "ItemCategoryTeamName": "category",
    "ItemCategoryTeamID": "Int64",
    "ItemCategoryGroupName": "category",
    "ItemCategoryGroupID": "Int64",
    "ItemCategoryAreaName": "category",
    "ItemCategoryAreaID": "Int64",
    "Brand": "category",
    # Stored as floats in the source file (0.0 / 1.0), not clean ints -- pandas
    # won't safely downcast float64 -> int8 during read_csv, so keep as float32.
    "eko": "float32",
    "organic": "float32",
    "krav": "float32",
    "fair_trade": "float32",
    "msc": "float32",
    "no_lactose": "float32",
}

df = pd.read_csv(
    DATA_PATH,
    dtype=dtypes,
    parse_dates=["DayDate"],
)

print(df.shape)
df.head()


## 2. Build receipt-level (basket) table

Channel behavior questions live at the basket/household level, not the line-item level.


In [ ]:
basket = df.groupby("receiptKey", observed=True).agg(
    customerId=("customerId", "first"),
    householdId=("householdId", "first"),
    store=("store", "first"),
    CoopOnlineYN=("CoopOnlineYN", "first"),
    DayDate=("DayDate", "first"),
    hourOfDay=("hourOfDay", "first"),
    n_lines=("ItemID", "count"),
    n_unique_items=("ItemID", "nunique"),
    total_qty=("quantity", "sum"),
    basket_value=("lineItemAmount", "sum"),
    basket_value_excl_vat=("lineItemAmountExclVat", "sum"),
    total_discount=("discountAmountExclVat", "sum"),
).reset_index()

# Label each basket's channel. Cast to plain str (not category) -- aggregating
# a categorical column with a set-returning function breaks, since pandas
# tries to rebuild the categorical output from unhashable set objects.
basket["channel"] = basket["CoopOnlineYN"].map({"Y": "Online", "N": "Offline"}).astype(str)

print(basket.shape)
basket.head()


In [ ]:
# Tag each household as offline-only / online-only / omni-channel,
# based on ALL their baskets across the 2 months
household_channels = basket.groupby("householdId", observed=True)["channel"].agg(set)

def label_channel_mix(s):
    if len(s) > 1:
        return "Omni-channel"
    return "Online-only" if "Online" in s else "Offline-only"

household_channel_mix = household_channels.apply(label_channel_mix)
household_channel_mix.value_counts()


In [ ]:
basket = basket.merge(
    household_channel_mix.rename("channel_mix"), on="householdId", how="left"
)
basket.head()


## 3. Basket value & size by channel

In [ ]:
channel_summary = basket.groupby("channel", observed=True).agg(
    n_baskets=("receiptKey", "count"),
    avg_basket_value=("basket_value", "mean"),
    median_basket_value=("basket_value", "median"),
    avg_items_per_basket=("n_lines", "mean"),
    avg_unique_items=("n_unique_items", "mean"),
).round(2)
channel_summary


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.barplot(data=basket, x="channel", y="basket_value", estimator="mean", errorbar=("ci", 95), ax=axes[0])
axes[0].set_title("Average basket value by channel")
axes[0].set_ylabel("Basket value (SEK, incl. VAT)")
axes[0].set_xlabel("")

sns.barplot(data=basket, x="channel", y="n_lines", estimator="mean", errorbar=("ci", 95), ax=axes[1])
axes[1].set_title("Average items per basket by channel")
axes[1].set_ylabel("Line items per basket")
axes[1].set_xlabel("")

plt.tight_layout()
plt.show()


In [ ]:
# Distribution, not just the mean -- means can hide a lot on retail data
fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(
    data=basket[basket["basket_value"].between(0, basket["basket_value"].quantile(0.99))],
    x="channel", y="basket_value", ax=ax,
)
ax.set_title("Basket value distribution by channel (99th percentile capped)")
ax.set_ylabel("Basket value (SEK)")
ax.set_xlabel("")
plt.show()


## 4. When do they shop? (time-of-day pattern)

In [ ]:
hourly = (
    basket.groupby(["channel", "hourOfDay"], observed=True)
    .size()
    .rename("n_baskets")
    .reset_index()
)
hourly["share"] = hourly.groupby("channel", observed=True)["n_baskets"].transform(lambda s: s / s.sum())

fig, ax = plt.subplots(figsize=(10, 5))
sns.lineplot(data=hourly, x="hourOfDay", y="share", hue="channel", marker="o", ax=ax)
ax.set_title("When customers shop, by channel (share of that channel's baskets)")
ax.set_xlabel("Hour of day")
ax.set_ylabel("Share of baskets")
plt.show()


### 4.1 Weekday vs. weekend timing, per channel

Same "when do they shop" question, but split by weekday vs. weekend to see if the online/offline
timing gap changes (e.g. offline shopping might concentrate around commute hours on weekdays but
spread out more on weekends).


In [ ]:
basket["dayOfWeek"] = basket["DayDate"].dt.dayofweek  # Monday=0 ... Sunday=6
basket["isWeekend"] = basket["dayOfWeek"] >= 5

def hourly_share_by_channel(data):
    hourly = (
        data.groupby(["channel", "hourOfDay"], observed=True)
        .size()
        .rename("n_baskets")
        .reset_index()
    )
    hourly["share"] = hourly.groupby("channel", observed=True)["n_baskets"].transform(lambda s: s / s.sum())
    return hourly

hourly_weekday = hourly_share_by_channel(basket[~basket["isWeekend"]])
hourly_weekend = hourly_share_by_channel(basket[basket["isWeekend"]])

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

sns.lineplot(data=hourly_weekday, x="hourOfDay", y="share", hue="channel", marker="o", ax=axes[0])
axes[0].set_title("Weekday: time of purchase by channel")
axes[0].set_xlabel("Hour of day")
axes[0].set_ylabel("Share of that channel's baskets")

sns.lineplot(data=hourly_weekend, x="hourOfDay", y="share", hue="channel", marker="o", ax=axes[1])
axes[1].set_title("Weekend: time of purchase by channel")
axes[1].set_xlabel("Hour of day")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()


## 5. Do online customers buy fewer "luxury" / premium items?

No direct luxury flag exists, so we build a proxy: **price per unit** (`lineItemAmountExclVat / quantity`).
Higher price-per-unit items are treated as more premium. We compare the price-per-unit distribution,
and the revenue share coming from the top price decile, across channels.


In [ ]:
lines = df[df["quantity"] > 0].copy()
lines["price_per_unit"] = lines["lineItemAmountExclVat"] / lines["quantity"]
lines["channel"] = lines["CoopOnlineYN"].map({"Y": "Online", "N": "Offline"})

# Define "premium" as top 10% price-per-unit across the whole dataset
premium_threshold = lines["price_per_unit"].quantile(0.90)
lines["is_premium"] = lines["price_per_unit"] >= premium_threshold

premium_share = lines.groupby("channel", observed=True)["is_premium"].mean().rename("premium_item_share")
premium_revenue_share = (
    lines.groupby(["channel", "is_premium"], observed=True)["lineItemAmountExclVat"].sum()
    .groupby(level=0).apply(lambda s: s / s.sum())
)

print(f"Premium threshold (price per unit): {premium_threshold:.2f} SEK\n")
print(premium_share, "\n")
print(premium_revenue_share)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.barplot(x=premium_share.index, y=premium_share.values, ax=ax)
ax.set_title("Share of line items that are 'premium' (top 10% price/unit), by channel")
ax.set_ylabel("Share of items")
ax.set_xlabel("")
plt.show()


## 6. Does being omni-channel mean spending more overall?

In [ ]:
household_totals = basket.groupby(["householdId", "channel_mix"], observed=True).agg(
    total_spend=("basket_value", "sum"),
    n_baskets=("receiptKey", "count"),
).reset_index()

household_summary = household_totals.groupby("channel_mix", observed=True).agg(
    n_households=("householdId", "count"),
    avg_total_spend=("total_spend", "mean"),
    avg_n_baskets=("n_baskets", "mean"),
).round(2)
household_summary


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.barplot(
    data=household_totals[household_totals["total_spend"].between(0, household_totals["total_spend"].quantile(0.99))],
    x="channel_mix", y="total_spend", estimator="mean", errorbar=("ci", 95), ax=axes[0],
)
axes[0].set_title("Avg. total spend per household (2 months)")
axes[0].set_ylabel("Total spend (SEK)")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=15)

sns.barplot(data=household_totals, x="channel_mix", y="n_baskets", estimator="mean", errorbar=("ci", 95), ax=axes[1])
axes[1].set_title("Avg. number of visits per household (2 months)")
axes[1].set_ylabel("Number of baskets")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()


## 7. How big is each group?

In [ ]:
mix_counts = household_channel_mix.value_counts()

fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(mix_counts.values, labels=mix_counts.index, autopct="%1.1f%%", startangle=90)
ax.set_title("Share of households by channel mix")
plt.show()


### 7.1 Corrected omni-channel rate (restricted to store-A shoppers)

Online orders only exist at **store A** -- store B has zero online transactions. That means any
household that only ever shops at store B can *never* be omni-channel, regardless of behavior.
The raw omni-channel % above is therefore understated by a purely structural factor, not a
behavioral one. Recomputing it only among households who shopped at store A at least once (i.e.
households who actually had the option to go omni) gives a fairer number.


In [ ]:
# Households that shopped at store A at least once -- the only ones who ever
# had the option to go online, and therefore the only ones who could be omni-channel
store_a_households = set(basket.loc[basket["store"] == "A", "householdId"])

eligible_mix = household_channel_mix[household_channel_mix.index.isin(store_a_households)]

print(f"Households eligible for omni (shopped at store A): {len(eligible_mix):,} "
      f"out of {household_channel_mix.shape[0]:,} total households\n")

print("Raw (all households):")
print(household_channel_mix.value_counts(normalize=True).round(4) * 100, "\n")

print("Corrected (store-A shoppers only):")
print(eligible_mix.value_counts(normalize=True).round(4) * 100)


In [ ]:
raw_pct = household_channel_mix.value_counts(normalize=True) * 100
corrected_pct = eligible_mix.value_counts(normalize=True) * 100

compare = pd.DataFrame({"Raw (all households)": raw_pct, "Corrected (store-A shoppers only)": corrected_pct}).fillna(0)

fig, ax = plt.subplots(figsize=(9, 5))
compare.plot(kind="bar", ax=ax)
ax.set_title("Omni-channel rate: raw vs. corrected for store-B customers who never had the option")
ax.set_ylabel("% of households")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()


## 8. Store A vs. Store B: income, and what store B is missing by not having online

Goal: compare the two stores' revenue and profit, then estimate how much of store A's profit is
attributable to (a) the online channel itself and (b) omni-channel households -- and use those
rates to project what store B might be leaving on the table by not offering online at all.

**Definitions:**
- `profit` = `lineItemAmountExclVat - lineItemCostExclVat` (Coop's margin, VAT excluded on both sides)
- Online/omni shares are computed **within store A only**, since store B has zero online transactions
  (see section 7.1) -- store B's numbers can't be split by channel, only used as the base to project onto.

**Important caveat (stated explicitly, not hidden in the numbers):** the "estimated missing profit"
for store B is a **naive linear extrapolation** -- it assumes store B's customers would adopt online
at the same rate, and generate the same profit mix, as store A's customers currently do. It ignores
demand differences between the two customer bases, the cost of standing up online fulfillment at
store B, and possible cannibalization of existing offline sales. Treat it as a directional "size of
the prize," not a forecast.


In [ ]:
# 8.1 Revenue & profit by store
df["channel"] = df["CoopOnlineYN"].map({"Y": "Online", "N": "Offline"}).astype(str)
df["profit"] = df["lineItemAmountExclVat"] - df["lineItemCostExclVat"]

store_summary = df.groupby("store", observed=True).agg(
    revenue_excl_vat=("lineItemAmountExclVat", "sum"),
    cost_excl_vat=("lineItemCostExclVat", "sum"),
    profit=("profit", "sum"),
)
store_summary["profit_margin_pct"] = (store_summary["profit"] / store_summary["revenue_excl_vat"] * 100).round(2)
store_summary = store_summary.round(0)
store_summary


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

store_summary[["revenue_excl_vat", "profit"]].plot(kind="bar", ax=axes[0])
axes[0].set_title("Revenue vs. profit by store")
axes[0].set_ylabel("SEK (2-month period)")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=0)

store_summary["profit_margin_pct"].plot(kind="bar", ax=axes[1], color="grey")
axes[1].set_title("Profit margin % by store")
axes[1].set_ylabel("Margin %")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()


### 8.2 What share of store A's profit comes from online sales?

In [ ]:
store_a_df = df[df["store"] == "A"]

channel_profit = store_a_df.groupby("channel", observed=True)["profit"].sum()
channel_profit_pct = (channel_profit / channel_profit.sum() * 100).round(2)

print("Store A profit by channel (SEK):")
print(channel_profit.round(0), "\n")
print("Store A profit share by channel (%):")
print(channel_profit_pct)

fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(channel_profit_pct.values, labels=channel_profit_pct.index, autopct="%1.1f%%", startangle=90)
ax.set_title("Store A: share of profit -- online vs. offline")
plt.show()


### 8.3 What share of store A's profit comes from omni-channel households?

In [ ]:
store_a_df = store_a_df.merge(household_channel_mix.rename("channel_mix"), on="householdId", how="left")

mix_profit = store_a_df.groupby("channel_mix", observed=True)["profit"].sum()
mix_profit_pct = (mix_profit / mix_profit.sum() * 100).round(2)

print("Store A profit by household channel mix (SEK):")
print(mix_profit.round(0), "\n")
print("Store A profit share by household channel mix (%):")
print(mix_profit_pct)

fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(mix_profit_pct.values, labels=mix_profit_pct.index, autopct="%1.1f%%", startangle=90)
ax.set_title("Store A: share of profit by household channel mix")
plt.show()


### 8.4 Naive projection: what could store B be missing?

Uses store A's online-to-offline and omni-to-non-omni profit *ratios* as a multiplier on store B's
actual profit. See the caveat above -- this is a directional estimate, not a forecast.


In [ ]:
store_b_profit = store_summary.loc["B", "profit"]

# Online / offline ratio, from store A
online_to_offline_ratio_a = channel_profit["Online"] / channel_profit["Offline"]
estimated_missing_online_profit_b = store_b_profit * online_to_offline_ratio_a

# Omni / non-omni ratio, from store A
non_omni_profit_a = mix_profit.sum() - mix_profit["Omni-channel"]
omni_to_non_omni_ratio_a = mix_profit["Omni-channel"] / non_omni_profit_a
estimated_missing_omni_profit_b = store_b_profit * omni_to_non_omni_ratio_a

print(f"Store A: online profit = {online_to_offline_ratio_a:.1%} of offline profit")
print(f"Store A: omni-channel-household profit = {omni_to_non_omni_ratio_a:.1%} of non-omni profit\n")

print(f"Store B current profit (2 months): {store_b_profit:,.0f} SEK\n")

print(f"Naive estimate, if store B matched store A's online/offline mix:")
print(f"  +{estimated_missing_online_profit_b:,.0f} SEK "
      f"(~{estimated_missing_online_profit_b / store_b_profit:.1%} uplift)\n")

print(f"Naive estimate, if store B matched store A's omni-channel mix:")
print(f"  +{estimated_missing_omni_profit_b:,.0f} SEK "
      f"(~{estimated_missing_omni_profit_b / store_b_profit:.1%} uplift)")


In [ ]:
scenarios = pd.Series({
    "Store B: current profit": store_b_profit,
    "Store B: + est. online uplift": store_b_profit + estimated_missing_online_profit_b,
    "Store B: + est. omni uplift": store_b_profit + estimated_missing_omni_profit_b,
})

fig, ax = plt.subplots(figsize=(8, 5))
scenarios.plot(kind="bar", ax=ax, color=["grey", "steelblue", "seagreen"])
ax.set_title("Store B: current vs. naive projected profit if it matched store A's channel mix")
ax.set_ylabel("Profit (SEK, 2-month period)")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.show()


### 8.5 Scenario: what if we converted customers until 75% were omni-channel?

Different question from 8.4: instead of projecting store B onto store A's *current* mix, this asks
"what if we actively pushed adoption up to a 75% omni-channel household base, company-wide?"

Method: compute each household's **total profit across both stores** (not just store A), average
it within each channel-mix group (Omni-channel vs. everyone else), then re-weight the total
household base to 75% omni / 25% non-omni and see what total profit that implies.

**Caveats (bigger ones than in 8.4):**
- Today's 113 omni-channel households are a tiny, self-selected group -- they may spend more
  *because* they're already high-engagement shoppers, not *because* they're omni-channel. Assuming
  a newly-converted household would match their average profit is optimistic.
- Ignores the cost of actually running a conversion program (incentives, marketing, discounts)
  needed to move ~75% of the customer base.
- Assumes total household count and non-omni behavior stay constant -- pure re-mix, not growth.


In [ ]:
# Total profit per household, across BOTH stores, labeled by their overall channel mix
df_with_mix = df.merge(household_channel_mix.rename("channel_mix"), on="householdId", how="left")

household_total_profit = (
    df_with_mix.groupby(["householdId", "channel_mix"], observed=True)["profit"]
    .sum()
    .reset_index()
)

# Simplify to two groups: Omni-channel vs. everyone else (Online-only + Offline-only combined)
household_total_profit["group"] = np.where(
    household_total_profit["channel_mix"] == "Omni-channel", "Omni-channel", "Non-omni"
)

group_stats = household_total_profit.groupby("group")["profit"].agg(
    n_households="count", avg_profit_per_household="mean"
)
group_stats


In [ ]:
total_households = household_total_profit["householdId"].nunique()
current_total_profit = household_total_profit["profit"].sum()
current_omni_share = group_stats.loc["Omni-channel", "n_households"] / total_households

avg_profit_omni = group_stats.loc["Omni-channel", "avg_profit_per_household"]
avg_profit_non_omni = group_stats.loc["Non-omni", "avg_profit_per_household"]

def projected_total_profit(omni_share, n_households=total_households):
    return n_households * (omni_share * avg_profit_omni + (1 - omni_share) * avg_profit_non_omni)

target_share = 0.75
projected_75 = projected_total_profit(target_share)
uplift_75 = projected_75 - current_total_profit

print(f"Total households: {total_households:,}")
print(f"Current omni-channel share: {current_omni_share:.2%}\n")

print(f"Avg profit / omni-channel household (2 months, both stores): {avg_profit_omni:,.0f} SEK")
print(f"Avg profit / non-omni household (2 months, both stores):     {avg_profit_non_omni:,.0f} SEK\n")

print(f"Current total profit (actual, 2 months): {current_total_profit:,.0f} SEK")
print(f"Projected total profit at 75% omni-channel adoption: {projected_75:,.0f} SEK")
print(f"Estimated uplift: {uplift_75:,.0f} SEK (~{uplift_75 / current_total_profit:.1%})")


In [ ]:
shares = np.linspace(0, 1, 21)
profits = [projected_total_profit(s) for s in shares]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(shares * 100, profits, marker="o")
ax.axvline(current_omni_share * 100, color="grey", linestyle=":", label=f"Current adoption ({current_omni_share:.1%})")
ax.axvline(75, color="red", linestyle="--", label="75% target")
ax.axhline(current_total_profit, color="grey", linestyle=":", alpha=0.5)
ax.set_xlabel("Omni-channel household share (%)")
ax.set_ylabel("Projected total profit (SEK, 2-month period)")
ax.set_title("Projected total profit vs. omni-channel adoption rate (naive re-mix, constant household count)")
ax.legend()
plt.tight_layout()
plt.show()


### 8.6 Controlling for buying power

One obvious confound for the 2.56x omni-vs-non-omni profit gap: maybe omni-channel households just
skew toward higher `DominantBuyingPowerClass`, and it's buying power driving the gap, not channel
behavior. Test this with **direct standardization**: compute the average profit within each
buying-power class separately for each group, then re-weight both groups using the *same*
(overall population) buying-power mix. If the gap shrinks a lot after this adjustment, buying power
was a real confound. If it barely moves, it isn't.


In [ ]:
# Attach each household's buying-power class (constant per household -- one MOSAIC profile per household)
household_bp = df.groupby("householdId")["DominantBuyingPowerClass"].first()
household_total_profit["bp"] = household_total_profit["householdId"].map(household_bp)

# Average profit within each buying-power class, split by omni vs. non-omni
strat_avg = household_total_profit.groupby(["bp", "group"], observed=True)["profit"].mean().unstack()
print("Avg profit per household, by buying-power class x group:")
print(strat_avg.round(0))
print()

# Overall population weights: share of ALL households in each buying-power class
pop_weights = household_total_profit["bp"].value_counts(normalize=True)
print("Population share by buying-power class:")
print(pop_weights.round(3))


In [ ]:
# Direct standardization: weight each stratum's avg profit by the OVERALL population's
# buying-power mix, so both groups are compared as if they had the same buying-power composition
standardized_avg = strat_avg.mul(pop_weights, axis=0).sum()

raw_avg = household_total_profit.groupby("group")["profit"].mean()

comparison = pd.DataFrame({
    "Raw avg profit / household": raw_avg,
    "Buying-power-adjusted avg profit / household": standardized_avg,
}).round(0)
comparison.loc["Omni / Non-omni multiplier"] = [
    (raw_avg["Omni-channel"] / raw_avg["Non-omni"]).round(2),
    (standardized_avg["Omni-channel"] / standardized_avg["Non-omni"]).round(2),
]
comparison


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
strat_avg.plot(kind="bar", ax=ax)
ax.set_title("Avg profit per household by buying-power class -- the omni gap holds within every class")
ax.set_ylabel("Avg profit per household (SEK, 2 months)")
ax.set_xlabel("Buying-power class")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()


In [ ]:
# Re-run the 75%-adoption projection (section 8.5) using the buying-power-adjusted
# averages instead of the raw ones -- this is the "fixed" table
avg_profit_omni_adj = standardized_avg["Omni-channel"]
avg_profit_non_omni_adj = standardized_avg["Non-omni"]

def projected_total_profit_adj(omni_share, n_households=total_households):
    return n_households * (omni_share * avg_profit_omni_adj + (1 - omni_share) * avg_profit_non_omni_adj)

rows = []
for target in [current_omni_share, 0.25, 0.50, 0.75, 1.00]:
    rows.append({
        "omni_share": target,
        "projected_profit_raw": projected_total_profit(target),
        "projected_profit_bp_adjusted": projected_total_profit_adj(target),
    })

projection_table = pd.DataFrame(rows).set_index("omni_share").round(0)
projection_table["difference"] = (
    projection_table["projected_profit_bp_adjusted"] - projection_table["projected_profit_raw"]
)
projection_table


### 8.7 Weekend opportunity: what if online ran on weekends at the same rate as weekdays?

Section 7's finding: online sales are **zero on weekends**, even at store A (verified directly --
this holds both store-wide and store-A-specific). This asks: if store A's online channel ran on
weekends at the same online/offline profit ratio it achieves on weekdays, how much extra profit
would that generate?

Method: compute store A's weekday online-to-offline profit ratio, then apply it as a multiplier to
store A's actual (currently offline-only) weekend profit -- same technique as the store B projection
in 8.4, just swapping the "store" dimension for a "day type" dimension.

**Caveats:** assumes weekend shoppers would adopt online at the same rate as weekday shoppers (untested --
weekend grocery trips may be more experience-driven, less suited to online); ignores the operational
cost of running weekend delivery/click-and-collect; ignores possible cannibalization of existing
weekend in-store sales.


In [ ]:
# Need isWeekend at line-item level for this (basket-level isWeekend was added in 4.1,
# but this section works off df directly since it needs store + channel + profit together)
df["isWeekend"] = df["DayDate"].dt.dayofweek >= 5
store_a_all = df[df["store"] == "A"]

weekday_profit_a = store_a_all[~store_a_all["isWeekend"]].groupby("channel", observed=True)["profit"].sum()
weekend_profit_a = store_a_all[store_a_all["isWeekend"]].groupby("channel", observed=True)["profit"].sum()

weekday_online_to_offline_ratio = weekday_profit_a["Online"] / weekday_profit_a["Offline"]
weekend_offline_profit = weekend_profit_a.get("Offline", 0)

print("Store A weekday profit by channel (SEK):")
print(weekday_profit_a.round(0), "\n")
print(f"Weekday online/offline profit ratio: {weekday_online_to_offline_ratio:.1%}\n")

print("Store A weekend profit by channel (SEK) -- actual:")
print(weekend_profit_a.round(0))


In [ ]:
estimated_missing_weekend_online_profit = weekend_offline_profit * weekday_online_to_offline_ratio
projected_weekend_total_profit = weekend_offline_profit + estimated_missing_weekend_online_profit

current_total_company_profit = df["profit"].sum()

print(f"Weekend current profit (store A, offline only): {weekend_offline_profit:,.0f} SEK")
print(f"Estimated missing weekend online profit: {estimated_missing_weekend_online_profit:,.0f} SEK")
print(f"Projected weekend total profit (with online): {projected_weekend_total_profit:,.0f} SEK")
print(f"Uplift over current weekend profit: {estimated_missing_weekend_online_profit / weekend_offline_profit:.1%}\n")

print(f"Current total company profit (both stores, 2 months): {current_total_company_profit:,.0f} SEK")
print(f"Estimated uplift as % of total company profit: "
      f"{estimated_missing_weekend_online_profit / current_total_company_profit:.1%}")


In [ ]:
scenarios = pd.Series({
    "Weekend: current (offline only)": weekend_offline_profit,
    "Weekend: + est. online (weekday ratio applied)": projected_weekend_total_profit,
})

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

scenarios.plot(kind="bar", ax=axes[0], color=["grey", "steelblue"])
axes[0].set_title("Store A weekend profit: current vs. naive projected (with online)")
axes[0].set_ylabel("Profit (SEK, 2-month period)")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=15)

pd.DataFrame({"Weekday": weekday_profit_a, "Weekend": weekend_profit_a.reindex(weekday_profit_a.index, fill_value=0)}).T.plot(
    kind="bar", stacked=True, ax=axes[1]
)
axes[1].set_title("Store A profit composition: weekday vs. weekend (actual)")
axes[1].set_ylabel("Profit (SEK)")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()


## 9. Takeaways (fill in after running)

- Basket value / size: ...
- Timing differences: ...
- Premium/luxury item share online vs. offline: ...
- Omni-channel spend premium: ...
- Household mix (how many are omni-channel today): ...
- Corrected omni-channel rate (store-A shoppers only) vs. raw rate: ...
- Store A vs. B profit/margin comparison: ...
- Share of store A's profit from online: ...
- Share of store A's profit from omni-channel households: ...
- Naive "size of the prize" for store B (online + omni uplift): ...
- Estimated profit uplift at 75% omni-channel adoption, company-wide: ...
- Is the omni-channel profit gap explained by buying power? (spoiler from manual check: no, ~2.5x gap holds after adjustment): ...
- Estimated weekend online opportunity, store A (manual check: ~830K SEK, ~91% uplift over current weekend profit, ~10% of total company profit): ...
